In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from msprep.multistateprep import STATE_CORE_COLS_REQUIRED


source_folder = "../../../data/CoMMpass/IA22/"
src = Path(source_folder)


## 1. Survival labels 

In [2]:
# Use the prepared survival table for over survival labels
surv = pd.read_csv(src / "MMRF_CoMMpass_IA22_STAND_ALONE_SURVIVAL.tsv", sep="\t")
surv["person_id"] = surv["PUBLIC_ID"].str.replace(r"^MMRF_", "", regex=True).astype(int)
surv = surv.rename(columns={"censos": "OSind", "ttcos": "OS"},)
surv = surv[surv.linesdy1 == 1]

surv = surv[["person_id", "OS", "OSind"]]


## 2. Baseline patient characteristics

In [3]:
# Use PER_PATIENT table for basic patient characteristics
per_patient = pd.read_csv(src / "MMRF_CoMMpass_IA22_PER_PATIENT.tsv", sep="\t")

per_patient["person_id"] = per_patient["PUBLIC_ID"].str.replace(r"^MMRF_", "", regex=True).astype(int)


per_patient = per_patient.rename(columns = {
    "D_PT_age": "age_at_baseline", 
    "D_PT_gender": "gender", 
    "D_PT_iss": "iss", 
    "line1sct": "sct",
    "D_PT_therclass": "trtclass",
})

per_patient["gender"] = per_patient["gender"] - 1

per_patient["height"] = pd.to_numeric(per_patient["demog_height"], errors="coerce") * per_patient["DEMOG_HEIGHTUNITOFM"].map({"in": 0.0254, "cm": 0.01})
per_patient["weight"] = pd.to_numeric(per_patient["demog_weight"], errors="coerce") * per_patient["DEMOG_WEIGHTUNITOFM"].map({"lb": 0.45359237, "kg": 1})
per_patient["height"] = np.where((per_patient.height < 1.20) | (per_patient.height > 2.10), np.nan, per_patient.height)
per_patient["weight"] = np.where((per_patient.weight < 30) | (per_patient.weight > 210), np.nan, per_patient.weight)
per_patient["bmi"] = per_patient["weight"] / (per_patient["height"]**2)

df_baseline = per_patient[["person_id", "age_at_baseline", "gender", "bmi", "iss", "sct", "trtclass"]].copy()


df_baseline = df_baseline.dropna(subset = ["iss"])
df_baseline["iss"] = df_baseline["iss"].astype(int)

df_baseline["trtclass"] = df_baseline.trtclass.map({
    'combined bortezomib/IMIDs/carfilzomib-based': 'BOR_IMiDs_CAR', 
    'IMIDs-based': 'IMiDs',
    'combined bortezomib/IMIDs-based': 'BOR_IMiDs', 
    'Bortezomib-based': 'BOR',
    'combined IMIDs/carfilzomib-based':'IMiDs_CAR' , 
    'Carfilzomib-based': 'CAR',
    'combined bortezomib/carfilzomib-based': 'BOR_CAR',
    'combined daratumumab/IMIDs/carfilzomib-based': 'DARA_IMiDs_CAR'
})


## 3. Visit-level data

In [4]:
# Lab/biomarker columns used as time-varying (state-level) features
state_features = {
 'D_LAB_cbc_abs_neut': 'pos',
 'D_LAB_chem_albumin': 'pos',
 'D_LAB_chem_calcium': 'pos',
 'D_LAB_chem_creatinine': 'pos',
 'D_LAB_cbc_hemoglobin': 'pos',
 'D_LAB_serum_m_protein': 'pos',
}

# prepare visit data
per_visit = pd.read_csv(src / "MMRF_CoMMpass_IA22_PER_PATIENT_VISIT.tsv", sep="\t", low_memory=False)
per_visit["person_id"] = per_visit["PUBLIC_ID"].str.replace(r"^MMRF_", "", regex=True).astype(int)


# restrict visits to patients who have a valid survival record 
per_visit = per_visit[per_visit.person_id.isin(surv.person_id)]

# keep only visits where treatment response was assessed as "Progressive Disease"
pd_visit = per_visit[(per_visit.AT_TREATMENTRESP == "Progressive Disease")].copy().sort_values("person_id")
pd_visit["state"] = 2  # state 2 = "First PD"
pd_visit = pd_visit.rename(columns={"AT_RESPONSEASSES": "first_PD"})

pd_visit["Tstart"] = pd_visit.first_PD
pd_visit = pd_visit.merge(surv, on="person_id", how="left")

# keep only PD events that occur after treatment start (Tstart > 0) and before the overall survival time (Tstart < OS) 
pd_visit = pd_visit[(pd_visit.Tstart > 0) & (pd_visit.Tstart < pd_visit.OS)]

# first PD per person
pd_visit = pd_visit.loc[pd_visit.groupby("person_id").Tstart.idxmin()]
pd_visit = pd_visit.rename(columns={"OS": "Tstop",})


pd_visit["event"] = np.where(pd_visit["OSind"] == 1, 3, 0)
pd_visit["time"] = pd_visit["Tstop"] - pd_visit["Tstart"]


# Patient/visit selection: keep only each patient's "Baseline" visit
base_visit = per_visit[(per_visit.VJ_INTERVAL == "Baseline")].copy().sort_values("person_id")
base_visit = base_visit.merge(surv, on="person_id", how="left")

base_visit["Tstart"] = 1
base_visit["state"] = 1

# merge both visit tables to identify the first event (death or PD)
base_visit = base_visit.merge(pd_visit[["person_id", "first_PD"]], on='person_id', how='left')

base_visit["event"] = np.where(base_visit.OSind == 1, 2, 0)
base_visit["event"] = np.where(base_visit.first_PD.isna(), base_visit.event, 1)
base_visit["Tstop"] = np.where(base_visit.first_PD.isna(), base_visit.OS, base_visit.first_PD)
base_visit["time"] = base_visit["Tstop"] - base_visit["Tstart"]

# reduce tables to core columns
base_visit = base_visit[STATE_CORE_COLS_REQUIRED + list(state_features.keys())].drop_duplicates()
pd_visit = pd_visit[STATE_CORE_COLS_REQUIRED + list(state_features.keys())].drop_duplicates()



df_states = pd.concat([base_visit, pd_visit]).sort_values(["person_id", "Tstart"])


# Fix time to start at 0
df_states["Tstop"] = df_states.Tstop - 1
df_states["Tstart"] = df_states.Tstart - 1
df_states["time"] = df_states.Tstop - df_states.Tstart



## 4. MultiStatePrep

In [ ]:


from msprep import MultiStatePrep

# Transition matrix 
tmat = np.array([
    [np.nan,      1,      2],   # Start -> PD (event 1), Start -> Death (event 2)
    [np.nan, np.nan,      3],   # PD -> Death (event 3)
    [np.nan, np.nan, np.nan],   # Death: no outgoing transitions (terminal)
])

baseline_features = {
       "age_at_baseline": "pos", 
       "gender": "bin", 
       "bmi": "pos", 
       "iss": "ord", 
       "sct": "bin",
       "trtclass": "cat",
       }


msp = MultiStatePrep.from_dataframes(
    dataset_name="commpass",
    baseline=df_baseline,
    states=df_states,
    baseline_features=baseline_features,
    state_features=state_features,
    state_names=["Treatment Start", "First PD", "Death"],
    tmat = tmat,
)

# Final patient/row selection
msp = msp.subset(
    dropna=True,
    truncate_at_first_gap=True,
)


#msp.save("data")
msp.event_summary()

